# 02 · ETL — Jewish Virtual Library (2025)

**Font:** Jewish Virtual Library — dades compilades del Israeli Central Bureau of Statistics (CBS)  
**Fitxer font:** `west_bank_settlements_population.csv`  
**Output:** `data/clean/jvl_long.csv`

---
## Rol en l'arquitectura del projecte

La JVL s'utilitza **únicament per completar l'any 2025**, que no està cobert per Peace Now.

| Font | Rang | Notebook |
|------|------|---------|
| Peace Now | 1993–2024 | 01_etl_settlements |
| **JVL** | **2025** | **02_etl_jvl** |

---
## Notes metodològiques

- Els noms dels assentaments ja venen normalitzats del CSV font (taula copiada manualment de JVL)
- No s'aplica cap `NAME_MAPPING` addicional en aquest notebook
- El merge final (notebook 03) pot requerir ajustos puntuals si algun nom no coincideix amb Peace Now
- `source = "jvl"` per coherència amb la versió CLEAN validada


## 1. Importació de llibreries

In [1]:
import pandas as pd
import os

print("Llibreries carregades correctament")

Llibreries carregades correctament


## 2. Càrrega del fitxer

In [2]:
df_raw = pd.read_csv("west_bank_settlements_population.csv")

print(f"Shape: {df_raw.shape}")
print(f"Columnes: {df_raw.columns.tolist()}")
df_raw.head(10)

Shape: (153, 10)
Columnes: ['Name', '2025', '2024', 'Year Founded', '2023', '2022', '2021', '2020', '2009', '1999']


,Name,2025,2024,Year Founded,2023,2022,2021,2020,2009,1999
0,Achya,9.0,NaN,2025.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Adam (Geva),6513.0,6333.0,1983.0,6226,5955.0,5837.0,5596.0,4157.0,707.0
2,Adura,628.0,598.0,1983.0,563,512.0,487.0,492.0,257.0,291.0
3,Adurayim,28.0,NaN,2025.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Alei (Elei) Zahav,5751.0,5588.0,1982.0,5358,4706.0,4213.0,3789.0,458.0,355.0
5,Alfei Menashe,8631.0,8645.0,1983.0,8671,8586.0,8622.0,8650.0,6718.0,4410.0
6,Almog,349.0,349.0,1977.0,314,313.0,302.0,313.0,153.0,156.0
7,Almon (Anatot),1524.0,1536.0,1982.0,1566,1546.0,1546.0,1483.0,827.0,672.0
8,Amichai,407.0,362.0,2018.0,318,273.0,232.0,213.0,NaN,NaN
9,Argeman,186.0,174.0,1968.0,171,163.0,165.0,163.0,166.0,155.0


## 3. Exploració inicial

In [3]:
print(df_raw.dtypes)
print(f"\nNuls per columna:")
print(df_raw.isnull().sum())

Name             object
2025            float64
2024            float64
Year Founded    float64
2023             object
2022            float64
2021            float64
2020            float64
2009            float64
1999            float64
dtype: object

Nuls per columna:
Name             0
2025             5
2024            17
Year Founded    10
2023            19
2022            22
2021            22
2020            24
2009            35
1999            39
dtype: int64


### 3.1 Control de qualitat: valors no numèrics

Abans de convertir les columnes d'any a numèric cal saber quins valors "estranys" hi ha,
perquè `pd.to_numeric(..., errors="coerce")` els convertirà silenciosament a `NaN`.
Aquí els llistem explícitament perquè quedi documentat *què* representa cada `NaN` del dataset net.

In [4]:
year_like_cols = [c for c in df_raw.columns if c not in ("Name", "Year Founded")]

print("Valors no numèrics detectats per columna:")
for col in year_like_cols + ["Year Founded"]:
    vals = df_raw[col].dropna().astype(str)
    non_numeric = vals[~vals.str.replace(",", "", regex=False).str.match(r"^-?\d+(\.0)?$")]
    if len(non_numeric):
        print(f"  {col}: {sorted(non_numeric.unique())}")

Valors no numèrics detectats per columna:
  2023: ['No Data']


**Resultat esperat:**
- Columna `2023` conté el text `"No Data"` per a *Ma'ale Shomron^* → es convertirà a `NaN` (absència real de dada, no un error).
- Columna `Year Founded` conté `"N/A"` per un parell d'assentaments (any de fundació desconegut) → no afecta aquest ETL perquè aquesta columna no s'utilitza.

Cap altra columna de població conté text no numèric.

### 3.2 Fila de totals
El CSV inclou una fila `Total` amb la suma agregada de tots els assentaments. S'ha d'eliminar abans de treballar amb dades per assentament (es fa a l'apartat 4).

In [5]:
print(df_raw[df_raw["Name"].str.lower() == "total"])

      Name      2025      2024  Year Founded    2023      2022      2021  \
152  Total  541085.0  530193.0           NaN  517407  490493.0  475481.0   

         2020      2009      1999  
152  463353.0  299340.0  176856.0  


### 3.3 Detecció d'anomalies

Es marquen els assentaments amb un salt de població molt gran (ràtio màxim/mínim > 3) entre 2020 i 2025.
La majoria corresponen a assentaments molt petits (poques desenes d'habitants), on un canvi de pocs
residents ja genera ràtios altes — no és necessàriament un error.

Val la pena documentar-ho explícitament, perquè si es propaga a l'anàlisi final (per exemple una taxa
de creixement) aquests casos poden distorsionar els resultats:

- **Kochav Yaakov**: 8.910 (2020) → 10.072 (2022) → 3.735 (2024). Salt i caiguda molt bruscos que no
  quadren amb l'evolució de la resta d'assentaments propers. Probablement un canvi en la delimitació
  administrativa de la localitat a la font original (CBS/JVL), no un èxode real. **Es manté tal qual**
  per fidelitat a la font, però es recomana tractar-lo amb precaució en anàlisis de tendència.
- **Mehola**: el valor de 2022 (`7304`) sembla una errada de transcripció (possiblement `730` amb un
  dígit de més). També es manté sense modificar.
- **Sa-Nur**: només té una dada (2025 = 14) i cap altre any. No és un error — Sa-Nur va ser evacuat el
  2005 i el valor de 2025 reflecteix una repoblació molt recent i puntual.

Aquestes observacions es deixen documentades aquí; **no es filtren ni es corregeixen** els valors,
perquè l'objectiu d'aquest notebook és una neteja fidel de la font, no una imputació d'outliers.

In [6]:
check_cols = ["2020", "2021", "2022", "2023", "2024", "2025"]
tmp = df_raw.copy()
for c in check_cols:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

anomalies = []
for _, row in tmp.iterrows():
    vals = row[check_cols].dropna()
    if len(vals) >= 2:
        ratio = vals.max() / max(vals.min(), 1)
        if ratio > 3:
            anomalies.append((row["Name"], ratio))

print("Assentaments amb ràtio màxim/mínim > 3 entre 2020-2025:")
for name, ratio in sorted(anomalies, key=lambda x: -x[1]):
    print(f"  {name}: ràtio {ratio:.1f}")

Assentaments amb ràtio màxim/mínim > 3 entre 2020-2025:
  Mehola: ràtio 11.5
  Avigael: ràtio 8.2
  Asael: ràtio 7.4
  Shacharit: ràtio 7.0
  Etz Efrayim^^: ràtio 3.6
  Mevuot Jericho: ràtio 3.4


## 4. Neteja

- Renombrar `Name` → `settlement` i `Year Founded` → `established` (per coherència amb el notebook de B'Tselem)
- Eliminar la fila `Total`
- Seleccionar només les columnes d'any dins l'abast (2020–2025) i renombrar-les amb el prefix `pop_`
- Convertir les columnes de població a numèric, deixant `NaN` on la font no tenia dada real (`"No Data"` inclòs)


In [7]:
df = df_raw.copy()

# Renombrar columnes
df = df.rename(columns={"Name": "settlement", "Year Founded": "established"})
# Eliminar símbols de nota al peu (^, ^^) que marquen fusions administratives
df["settlement"] = df["settlement"].str.replace(r"[\^*]", "", regex=True).str.strip()

# Eliminar fila de totals
df = df[df["settlement"].notna()]
df = df[~df["settlement"].str.lower().str.contains("total", na=False)]

# Seleccionar només l'abast temporal 2020-2025 i renombrar amb prefix pop_
target_years = ["2020", "2021", "2022", "2023", "2024", "2025"]
rename_map = {y: f"pop_{y}" for y in target_years}
df = df.rename(columns=rename_map)
pop_cols = list(rename_map.values())

# Convertir columnes de població a numèric ("No Data" i similars -> NaN)
for col in pop_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")

df_clean = df[["settlement"] + pop_cols].copy()

print(f"Shape després neteja: {df_clean.shape}")
df_clean.head()

Shape després neteja: (152, 7)


,settlement,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024,pop_2025
0,Achya,NaN,NaN,NaN,NaN,NaN,9.0
1,Adam (Geva),5596.0,5837.0,5955.0,6226.0,6333.0,6513.0
2,Adura,492.0,487.0,512.0,563.0,598.0,628.0
3,Adurayim,NaN,NaN,NaN,NaN,NaN,28.0
4,Alei (Elei) Zahav,3789.0,4213.0,4706.0,5358.0,5588.0,5751.0


## 5. Transformació a format llarg (melt)

In [8]:
df_long = df_clean.melt(
    id_vars=["settlement"],
    value_vars=pop_cols,
    var_name="year_raw",
    value_name="population"
)

# Extreure any
df_long["year"] = df_long["year_raw"].str.extract(r"(\d{4})").astype(int)

# ── Filtrar únicament l'any 2025 ──
# Peace Now cobreix 1993–2024. La JVL s'utilitza exclusivament per completar el 2025.
df_long = df_long[df_long["year"] == 2025]

# Eliminar files sense població (NaN reals de la font)
df_long = df_long.dropna(subset=["population"])
df_long["population"] = df_long["population"].astype(int)

# Eliminar assentaments amb població 0
df_long = df_long[df_long["population"] > 0]

# Afegir font
df_long["source"] = "jvl"

# year_raw s'elimina implícitament en seleccionar les columnes finals
df_long = df_long[["settlement", "year", "population", "source"]]
df_long = df_long.sort_values("settlement").reset_index(drop=True)

print(f"Shape final: {df_long.shape}")
print(f"Any cobert: {df_long['year'].unique()}")
print(f"Assentaments únics: {df_long['settlement'].nunique()}")
df_long.head(10)

Shape final: (147, 4)
Any cobert: [2025]
Assentaments únics: 147


,settlement,year,population,source
0,Achya,2025,9,jvl
1,Adam (Geva),2025,6513,jvl
2,Adura,2025,628,jvl
3,Adurayim,2025,28,jvl
4,Alei (Elei) Zahav,2025,5751,jvl
5,Alfei Menashe,2025,8631,jvl
6,Almog,2025,349,jvl
7,Almon (Anatot),2025,1524,jvl
8,Amichai,2025,407,jvl
9,Argeman,2025,186,jvl


## 5. Transformació a format llarg (melt)

## 6. Verificació

In [9]:
print(f"Any cobert: {df_long['year'].unique()}")
print(f"Assentaments únics: {df_long['settlement'].nunique()}")
print(f"Població total documentada 2025: {df_long['population'].sum():,}")
print(f"\nTop 10 per població:")
print(df_long.nlargest(10, "population")[["settlement", "population"]].to_string(index=False))

Any cobert: [2025]
Assentaments únics: 147
Població total documentada 2025: 541,085

Top 10 per població:
    settlement  population
  Modi'in Ilit       92339
  Beitar Illit       74760
 Ma'ale Adumim       40800
   Givat Ze'ev       25630
         Ariel       22273
         Efrat       13502
Karnei Shomron       11199
        Oranit       10212
 Shaar Shomron        9024
 Alfei Menashe        8631


In [10]:
# Exemple: un assentament conegut
exemple = df_long[df_long["settlement"].str.contains("Beitar", na=False)]
print(exemple)

      settlement  year  population source
19  Beitar Illit  2025       74760    jvl


### 6.1 Nota per al notebook de merge (03)

Alguns noms d'assentament d'aquesta font porten símbols de nota al peu (`^`, `^^`) que marquen
fusions administratives (p. ex. `Karnei Shomron^`, `Shaar Shomron^^`, que va absorbir *Etz Efrayim*
i *Shaare Tikva* el 2022). No s'han normalitzat aquí perquè fer-ho requereix una decisió de mapeig
explícita respecte als noms de B'Tselem, que pertany al notebook de merge, no a aquest.


## 7. Exportació

In [11]:
os.makedirs("data/clean", exist_ok=True)

output_path = "data/clean/jvl_long.csv"
df_long.to_csv(output_path, index=False)

print(f"✓ Exportat: {output_path}")
print(f"  Registres:    {df_long.shape[0]}")
print(f"  Any cobert:   2025")
print(f"  Assentaments: {df_long['settlement'].nunique()}")
print(f"  Columnes:     {list(df_long.columns)}")
print(f"\nAquest CSV entrarà al notebook 03_merge_population per completar la sèrie 1993–2025.")

✓ Exportat: data/clean/jvl_long.csv
  Registres:    147
  Any cobert:   2025
  Assentaments: 147
  Columnes:     ['settlement', 'year', 'population', 'source']

Aquest CSV entrarà al notebook 03_merge_population per completar la sèrie 1993–2025.
